# 06 - Baseline Models under the Corrected Protocol (P0)

**Purpose:** Fit all models under P0, storing MAE/RMSE/sMAPE/MASE and skill vs seasonal-naive, repeated across seeds and horizons.

**Inputs:** Feature matrices (notebook 04); validated P0 splitter (notebook 05).

**Outputs:** results/metrics/ P0 baseline metrics table.

**Runtime estimate:** 15-60 minutes depending on profile (fast vs full).

## Load experiment configuration and select profile (fast/full)

In [1]:
import pandas as pd

from sdg_leakage_audit.utils.io import read_yaml
from sdg_leakage_audit.utils.paths import configs_dir, results_metrics_dir, to_project_relative

experiment_cfg_raw = read_yaml(configs_dir() / "experiment.yaml")
datasets_cfg = read_yaml(configs_dir() / "datasets.yaml")["datasets"]

ACTIVE_PROFILE = "fast"  # Phase-2 implementation run; "full" is for the final results
profile = experiment_cfg_raw["profiles"][ACTIVE_PROFILE]
run_config = {
    "data_subsample_frac": profile["data_subsample_frac"],
    "model_param_grid": profile["model_param_grid"],
}
seeds = profile["seeds"]

print(f"Active profile: {ACTIVE_PROFILE!r} -> {profile['description']}")
print(f"run_config = {run_config}, seeds = {seeds}")


Active profile: 'fast' -> Smoke-test profile: minimal grid, 1-2 seeds, subsampled data, for pipeline validation.
run_config = {'data_subsample_frac': 0.05, 'model_param_grid': 'params_fast'}, seeds = [0]


## Fit seasonal-naive and persistence baselines

In [2]:
from sdg_leakage_audit.evaluation.runner import run_single_cell

results_rows = []

for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["seasonal_naive", "persistence"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"Baselines done: {len(results_rows)} cells (seasonal_naive + persistence).")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean()


2026-10-20 00:21:15 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:17 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:19 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:21 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:23 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


...

2026-10-20 00:21:30 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:31 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:31 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


Baselines done: 30 cells (seasonal_naive + persistence).


mae       smape
dataset_id                    model_name                           
air_quality_uci               persistence      0.443333   38.066075
                              seasonal_naive   0.572500   42.075785
appliances_energy_prediction  persistence     85.211506   43.761068
                              seasonal_naive  66.142132   38.414691
beijing_multisite_air_quality persistence     45.708429   60.699613
                              seasonal_naive  79.148137  103.683283
household_power_consumption   persistence      0.398118   30.346537
                              seasonal_naive   0.724391   54.240431
jena_climate                  persistence      1.162768   37.986620
                              seasonal_naive   2.806254   78.995804

## Fit Ridge, RandomForest, LightGBM, MLP

In [3]:
for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["ridge", "random_forest", "lightgbm", "mlp"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"ML models done: {len(results_rows)} cells total so far.")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean().loc[
    pd.IndexSlice[:, ["ridge", "random_forest", "lightgbm", "mlp"]], :
]


2026-10-20 00:21:32 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:33 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:41 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:44 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:21:56 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


...

2026-10-20 00:24:12 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:15 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:15 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


ML models done: 90 cells total so far.


,,mae,smape
dataset_id,model_name,,
air_quality_uci,ridge,0.991308,107.487290
appliances_energy_prediction,ridge,128.401386,93.528603
beijing_multisite_air_quality,ridge,43.734627,70.172074
household_power_consumption,ridge,0.415103,36.824600
jena_climate,ridge,1.495416,42.417149
air_quality_uci,random_forest,0.947482,61.200144
appliances_energy_prediction,random_forest,75.685752,42.846490
beijing_multisite_air_quality,random_forest,56.631058,67.264527
household_power_consumption,random_forest,0.431579,31.400216


## Fit small LSTM/GRU

In [4]:
for dataset_id, cfg in datasets_cfg.items():
    for horizon in cfg["forecast_horizons"]:
        for seed in seeds:
            for model_name in ["lstm", "gru"]:
                result = run_single_cell(dataset_id, "P0", model_name, horizon, seed, run_config)
                results_rows.append(result)

print(f"All models done: {len(results_rows)} cells total.")
pd.DataFrame(results_rows).groupby(["dataset_id", "model_name"])[["mae", "smape"]].mean().loc[
    pd.IndexSlice[:, ["lstm", "gru"]], :
]


2026-10-20 00:24:24 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:29 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:32 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:35 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:37 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


...

2026-10-20 00:24:49 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:50 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


2026-10-20 00:24:51 | INFO     | sdg_leakage_audit.utils.seed | Global random seed set to 0


All models done: 120 cells total.


,,mae,smape
dataset_id,model_name,,
air_quality_uci,lstm,1.219008,200.000000
appliances_energy_prediction,lstm,129.021961,199.221910
beijing_multisite_air_quality,lstm,91.704178,198.225618
household_power_consumption,lstm,1.343295,200.000000
jena_climate,lstm,4.808719,194.229213
air_quality_uci,gru,0.978021,144.421037
appliances_energy_prediction,gru,128.944316,198.796871
beijing_multisite_air_quality,gru,91.562570,195.787625
household_power_consumption,gru,1.111792,132.911408


## Aggregate P0 metrics across seeds/horizons

In [5]:
p0_metrics_df = pd.DataFrame(results_rows)

out_path = results_metrics_dir() / "p0_baseline_metrics.csv"
p0_metrics_df.to_csv(out_path, index=False)
print(f"Saved {len(p0_metrics_df)} rows -> {to_project_relative(out_path)}")

# Skill vs seasonal-naive, averaged across horizons/seeds, per dataset x model --
# the headline P0 baseline summary table for the rest of this project to build on.
skill_summary = (
    p0_metrics_df[p0_metrics_df["model_name"] != "seasonal_naive"]
    .groupby(["dataset_id", "model_name"])["skill_score"]
    .mean()
    .unstack("model_name")
    .round(3)
)
skill_summary


Saved 120 rows -> results\metrics\p0_baseline_metrics.csv


model_name,gru,lightgbm,lstm,mlp,persistence,random_forest,ridge
dataset_id,,,,,,,
air_quality_uci,-0.710,-0.633,-1.158,-1.954,0.248,-0.877,-0.775
appliances_energy_prediction,-0.951,-0.081,-0.952,-0.059,-0.313,-0.153,-0.950
beijing_multisite_air_quality,-0.159,0.350,-0.160,0.311,0.431,0.295,0.455
household_power_consumption,-0.535,0.399,-0.854,0.279,0.451,0.405,0.427
jena_climate,-0.701,0.286,-0.714,0.382,0.585,0.347,0.466


## Findings / what to carry forward

- All 8 models (seasonal-naive, persistence, Ridge, RandomForest, LightGBM, MLP, LSTM, GRU)
  were fit under the validated P0 protocol across all 5 datasets and each dataset's configured
  forecast horizons, using the `fast` profile (seed 0, 5% of each dataset's most-recent data,
  `params_fast` hyperparameters) -- a pipeline-correctness smoke test, not final numbers.
- Results saved to `results/metrics/p0_baseline_metrics.csv`; `skill_score` (fraction of
  seasonal-naive's error eliminated) is the headline comparison metric carried into notebook 07,
  where the same models are re-run under P1-P6 to measure leakage-driven inflation relative to
  these P0 numbers.
- Evaluation framing: "direct" horizon regression (each row's causal features predict the
  target `horizon` steps ahead of that row, evaluated independently per row) rather than a
  single-origin recursive multi-step forecast -- simpler to implement correctly and leakage-safe,
  at the cost of not modeling error accumulation across steps the way a recursive forecast would.
- Known approximation: the purge gap enforced by `chronological_split_with_gap` is `horizon`
  *positions* in the feature matrix after NaN-dropping, not strictly `horizon` *time steps*,
  wherever upstream missingness caused a few interior rows to be dropped. This is negligible
  for these datasets (missingness rates are low relative to horizon sizes) but is a known
  simplification worth stating explicitly rather than silently assuming exactness.
- LSTM/GRU operate on a short sequence built from each dataset's lag-feature columns (ordered by
  increasing lag), not a full sliding-window of raw observations -- a deliberate Phase-2 scope
  simplification; a true windowed-sequence recurrent model is a natural `full`-profile extension.
- Next (notebook 07): re-run this exact model/dataset/horizon grid under P1-P6 and compare against
  these P0 numbers to compute inflation.
